## 步骤1: 安装 haveged

In [ ]:
%%bash
echo "Installing haveged..."
sudo pacman -S --noconfirm --needed haveged >/dev/null
sudo systemctl enable --now haveged >/dev/null

## 步骤2: 安装 VirtualBox

In [ ]:
%%bash
echo "Installing virtualbox..."
sudo pacman -S --noconfirm --needed virtualbox \
	linux$(uname -r | cut -d. -f1-2 | tr -d . | head -c3)-virtualbox-host-modules \
	virtualbox-ext-vnc >/dev/null

for mod in vboxdrv vboxnetadp vboxnetflt; do
	sudo modprobe "$mod" >/dev/null && echo "  ✓ Loaded $mod" || echo "  ✗ Failed to load $mod"
done
sudo usermod -aG vboxusers "$USER" && echo "  ✓ Added $USER to vboxusers group"

## 步骤3: 安装 Docker

In [ ]:
%%bash
echo "Installing docker..."
sudo pacman -S --noconfirm --needed docker >/dev/null

### 配置 Docker

In [ ]:
%%bash
daemon_file="/etc/docker/daemon.json"

sudo systemctl stop docker &>/dev/null

sudo mkdir -p /etc/docker
sudo tee "$daemon_file" > /dev/null <<'EOF'
{
  "registry-mirrors": [
    "https://docker.1ms.run",
    "https://docker.1panel.live",
    "https://docker.m.ixdev.cn",
    "https://hub.rat.dev",
    "https://dockerproxy.net",
    "https://docker.hlmirror.com",
    "https://hub1.nat.tf",
    "https://hub3.nat.tf",
    "https://docker.m.daocloud.io",
    "https://docker.kejilion.pro",
    "https://hub.1panel.dev",
    "https://dockerproxy.cool",
    "https://proxy.vvvv.ee"
  ]
}
EOF

sudo usermod -aG docker "$USER" &>/dev/null

sudo systemctl daemon-reload
sudo systemctl enable --now docker &>/dev/null

echo "✓ Docker configured"
echo "⚠ Please logout and login again for group changes to take effect"

## 步骤4: 安装 aria2

In [ ]:
%%bash
echo "Installing aria2..."
sudo pacman -S --noconfirm --needed aria2 >/dev/null

## 步骤5: 安装 Miniconda3

### 配置 pip

In [ ]:
%%bash
src="$HOME/.config/pip"
mkdir -p $src
cat > "$src/pip.conf" <<'EOF'
[global]
index-url = https://pypi.mirrors.ustc.edu.cn/simple/
trusted-host = pypi.mirrors.ustc.edu.cn
timeout = 120
EOF
echo "✓ pip config created"

### 安装 Miniconda3

In [ ]:
%%bash
echo "Installing miniconda3..."
install_dir="/data/.path/.miniconda3"

if [[ -f "$install_dir/bin/conda" ]]; then
	echo "✓ Miniconda3 already installed, skipping"
else
	save_path="$HOME/Downloads"
	cd "$save_path"
	download_url="https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh"
	installer="Miniconda3-latest-Linux-x86_64.sh"
	aria2c -x 16 -s 16 "$download_url" -o "$installer"
	
	rm -rf "$install_dir"
	mkdir -p "$(dirname "$install_dir")"
	bash "$installer" -b -f -p "$install_dir"
	
	conda_bin="$install_dir/bin/conda"
	
	if [[ -f "$conda_bin" ]]; then
		echo "✓ Miniconda3 installed successfully"
		"$conda_bin" init zsh
		"$conda_bin" init bash
		"$conda_bin" config --set auto_activate_base false
		"$conda_bin" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
		"$conda_bin" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
		"$conda_bin" install jupyter ipykernel -y
		echo "✓ Miniconda3 configured!"
	else
		echo "❌ Miniconda3 installation failed"
	fi
fi

## 步骤6: 安装 Ardour

In [ ]:
%%bash
echo "Installing ardour..."
sudo pacman -S --noconfirm --needed ardour >/dev/null

sudo mkdir -p /etc/security/limits.d
sudo tee "/etc/security/limits.d/$USER-audio-unlimited.conf" >/dev/null <<EOF
@audio   -  rtprio     95
@audio   -  memlock    unlimited
EOF
sudo usermod -aG audio "$USER"


# LANGUAGE=zh_CN.UTF-8 ardour8

## 步骤7: 安装 zsh-sudo 插件

In [ ]:
%%bash
echo "Installing zshplugins zsh-sudo..."
zsh_plugins_dir="/data/.zshplugins" && mkdir -p "$zsh_plugins_dir"
zsh_sudo_dir="$zsh_plugins_dir/zsh-sudo"

if [[ ! -d "$zsh_sudo_dir" ]]; then
	git clone https://github.com/none9632/zsh-sudo.git "$zsh_sudo_dir" >/dev/null
	echo "  ✓ Cloned zsh-sudo"
fi

if ! grep -q "zsh-sudo.zsh" ~/.zshrc 2>/dev/null; then
	echo "source $zsh_sudo_dir/zsh-sudo.zsh" >>~/.zshrc
	echo "  ✓ Added zsh-sudo to .zshrc"
fi

## 步骤8: 安装 mpv

In [ ]:
%%bash
echo "Installing mpv..."
sudo pacman -S --noconfirm --needed mpv ffmpeg >/dev/null

## 步骤9: 安装 Telegram

In [ ]:
%%bash
echo "Installing telegram-desktop..."
sudo pacman -S --noconfirm --needed telegram-desktop >/dev/null

## 步骤10: 安装 OBS Studio

In [ ]:
%%bash
echo "Installing obs-studio..."
sudo pacman -S --noconfirm --needed obs-studio >/dev/null

## 步骤11: 安装 qBittorrent

In [ ]:
%%bash
echo "Installing qbittorrent..."
sudo pacman -S --noconfirm --needed qbittorrent >/dev/null

## 步骤12: 安装 Ventoy

In [ ]:
%%bash
echo "Installing ventoy..."
sudo pacman -S --noconfirm --needed ventoy >/dev/null

## 步骤13: 安装 Wine

In [ ]:
%%bash
echo "Installing wine..."
sudo pacman -S --noconfirm --needed wine wine-mono wine-gecko winetricks >/dev/null

## 步骤14: 安装 yabridge

In [ ]:
%%bash
echo "Installing yabridge..."
sudo pacman -S --noconfirm --needed yabridgectl >/dev/null

## 步骤15: 添加 AppImage 路径到 PATH

In [ ]:
%%bash
tgt="/data/.path/.appimage"
mkdir -p "$tgt"
e_l="export PATH=\"$tgt:\$PATH\""

if grep -qF "$e_l" "$HOME/.zshrc"; then
	echo "  ✓ PATH already contains $tgt"
else
	echo "$e_l" >>"$HOME/.zshrc"
	echo "  ✓ Added $tgt to PATH"
fi

## backup

In [ ]:
%%bash
snapshot_dir="/.snapshots"
sudo mkdir -p "$snapshot_dir"
snapshot_name="$snapshot_dir/soft_basic"
if [[ -d "$snapshot_name" ]]; then
    echo "  ✓ Snapshot already exists: $snapshot_name"
else
    sudo btrfs subvolume snapshot -r / "$snapshot_name"
fi